In [ ]:
# ============================================================
# IMPORTS
# ============================================================
from graph import app

import os
import time
import pandas as pd
import torch

from tqdm.auto import tqdm
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI

load_dotenv(override=True)

In [ ]:
# ============================================================
# Experiment Configuration
# ============================================================

# Dataset: "movies" or "multidb"
EXPERIMENT = "movies"

# Model
MODEL_ID = "meta-llama/llama-3.1-8b-instruct"
# MODEL_ID = "meta-llama/llama-3.2-3b-instruct"
# MODEL_ID = "qwen/qwen-2.5-7b-instruct"

MODEL_LABEL = "llama_3_1_8b"

# Filename label used by the existing experiment outputs
MODEL_FILE_LABELS = {
    "llama_3_1_8b": "llama_3_1_8b",
    "qwen_2_5_7b": "qwen_2_5_7b",
    "llama_3_2_3b": "llama_3_2_3b",
}

MODEL_FILE_LABEL = MODEL_FILE_LABELS[MODEL_LABEL]

# IMPORTANT:
# CONFIGURATION is used only for output naming
# It does NOT enable/disable memory mechanisms
# For ablation runs, manually configure the corresponding
# Generator/Corrector memory calls before execution

# Experimental configuration
# CONFIGURATION = "BASELINE"
# CONFIGURATION = "HISTORY_ONLY"
# CONFIGURATION = "TRAJECTORY_ONLY"
# CONFIGURATION = "TRAJECTORY_HISTORY"
# CONFIGURATION = "GEN_HELPER_HISTORY"
# CONFIGURATION = "GEN_HELPER_TRAJ"
CONFIGURATION = "MEMORY_ALL_IN"

RUN_NAME = f"{MODEL_LABEL}_{CONFIGURATION}"

CONFIG_FILE_SUFFIXES = {
    "BASELINE": "baseline",
    "HISTORY_ONLY": "history_only",
    "TRAJECTORY_ONLY": "traj_only",
    "TRAJECTORY_HISTORY": "traj_history",
    "GEN_HELPER_HISTORY": "gen_helper_history",
    "GEN_HELPER_TRAJ": "gen_helper_traj",
    "MEMORY_ALL_IN": "mem_all_in",
}

CONFIG_FILE_SUFFIX = CONFIG_FILE_SUFFIXES[CONFIGURATION]

OUTPUT_NAME = f"{MODEL_FILE_LABEL}_{CONFIG_FILE_SUFFIX}"

# Dataset paths

if EXPERIMENT == "movies":
    MEMORY_DATASET = "Movies_DB_Datasets/50_memory_queries_thesis.xlsx"
    TEST_DATASET = "Movies_DB_Datasets/100_test_queries_thesis.xlsx"

    OUTPUT_ROOT = "Outputs_Evaluation/Movies_Experiment"

elif EXPERIMENT == "multidb":
    MEMORY_DATASET = "Multi_DBs_Datasets/250_memory_queries_thesis.xlsx"
    TEST_DATASET = "Multi_DBs_Datasets/500_test_queries_thesis.xlsx"

    OUTPUT_ROOT = "Outputs_Evaluation/MultiDB_Experiment"

else:
    raise ValueError("EXPERIMENT must be either 'movies' or 'multidb'")

print(f"Experiment: {EXPERIMENT}")
print(f"Model:      {MODEL_ID}")
print(f"Run:        {RUN_NAME}")
print(f"Memory set: {MEMORY_DATASET}")
print(f"Test set:   {TEST_DATASET}")

In [ ]:
# ============================================================
# Model Initialization
# ============================================================

print("\nSystem Initialization")

if not os.getenv("OPENROUTER_API_KEY"):
    raise ValueError("OPENROUTER_API_KEY is missing.")

print(f"Initializing OpenRouter model: {MODEL_ID}...")

model = ChatOpenAI(
    model=MODEL_ID,
    api_key=os.getenv("OPENROUTER_API_KEY"),
    base_url="https://openrouter.ai/api/v1",
    temperature=0.0,
    max_tokens=512,
    extra_body={
        "provider": {
            "allow_fallbacks": True,
        }
    }
)

print(f"Model {MODEL_ID} loaded successfully.")

In [ ]:
def build_inputs(row, question_text, save_to_memory, experiment):
    """
    Build graph inputs for the selected experimental dataset.
    """

    if experiment == "movies":
        return {
            "question": question_text,
            "database_url": os.getenv("NEO4J_URI"),
            "database_name": "movies",
            "database_user": os.getenv("NEO4J_USERNAME"),
            "database_password": os.getenv("NEO4J_PASSWORD"),
            "save_to_memory": save_to_memory,
        }

    if experiment == "multidb":
        return {
            "question": question_text,
            "database_url": row["url"],
            "database_name": row["database_name"],
            "database_user": row["username"],
            "database_password": row["password"],
            "save_to_memory": save_to_memory,
        }

    raise ValueError(f"Unsupported experiment: {experiment}")

In [ ]:
def evaluate_model(run_name, output_name, configuration, model, target_file, output_root, experiment, save_to_memory=False):
    '''
    Evaluates the model using a specific Excel dataset.
    Uses the database connection information provided per row and passes 'save_to_memory'
    to prevent test data leakage during benchmark runs.
    '''

    # Load target Excel file
    file_path = target_file
    df = pd.read_excel(file_path)

   # Final Excel results are grouped by experimental configuration
    output_dir = os.path.join(output_root, configuration)
    os.makedirs(output_dir, exist_ok=True)

    # Temporary checkpoint files kept separately
    temp_dir = os.path.join(output_root, "temp")
    os.makedirs(temp_dir, exist_ok=True)

    target_name = os.path.splitext(os.path.basename(target_file))[0]

    # Keep the og experiment filename convention
    target_name = target_name.replace("_queries_thesis", "")

    results_path_xlsx = os.path.join(
        output_dir,
        f"{target_name}_{output_name}.xlsx"
    )

    results_path_pkl = os.path.join(
        temp_dir,
        f"{target_name}_{output_name}.pkl"
    )

    # Resume progress if file already exists
    if os.path.exists(results_path_xlsx):
        print(f"Found existing progress at {results_path_xlsx}. Resuming...")
        df_results = pd.read_excel(results_path_xlsx).astype(object)
    else:
        df_results = df.copy().astype(object)

        new_cols = [
            'generated_cypher', 'answer', 'path', 'output',
            'cypher_errors_history', 'cypher_errors', 'is_KG_Valid_correct',
            'is_KG_Valid_correct_with_full_schema', 'total_tokens',
            'completion_tokens', 'prompt_tokens', 'eval_time_s'
        ]

        for col in new_cols:
            if col not in df_results.columns:
                df_results[col] = None

    config = {
        'configurable': {
            'model': model,
            'model_name': run_name,
            'attempts_w_filtered': 5,
            'total_attempts': 10
        },
        'recursion_limit': 50
    }

    print(f"Starting execution on {target_file} | Memory Saving: {save_to_memory}")

    # Main evaluation loop
    pbar = tqdm(
        df_results.iterrows(),
        total=df_results.shape[0],
        desc="Evaluating queries",
        unit="query"
    )

    for i, row in pbar:
        pbar.set_postfix(
            db=row['database_name'],
            query=i
        )

        # Skip if already computed
        if pd.isna(row['generated_cypher']) or row['generated_cypher'] is None:

            # Identify question column
            question_text = row['question'] if 'question' in row else row['Question']

            inputs = build_inputs(
                row=row,
                question_text=question_text,
                save_to_memory=save_to_memory,
                experiment=experiment
            )

            start = time.perf_counter()
            result = app.invoke(inputs, config=config)
            elapsed = time.perf_counter() - start

            # Extract Graph State Results
            path_list = result.get('path', [])
            is_valid_correct = result.get('is_KG_Valid_correct', False)

            is_full_schema_correct = ('schema_retriever' in path_list) and bool(is_valid_correct)

            # Store All Metrics
            df_results.at[i, 'eval_time_s'] = round(elapsed, 4)
            df_results.at[i, 'generated_cypher'] = result.get('cypher_statement', '')
            df_results.at[i, 'output'] = str(result.get('database_records', ''))
            df_results.at[i, 'path'] = str(path_list)
            df_results.at[i, 'cypher_errors'] = str(result.get('cypher_errors', []))
            df_results.at[i, 'cypher_errors_history'] = str(result.get('cypher_errors_history', []))
            df_results.at[i, 'is_KG_Valid_correct'] = is_valid_correct
            df_results.at[i, 'is_KG_Valid_correct_with_full_schema'] = is_full_schema_correct
            df_results.at[i, 'answer'] = result.get('answer', '')
            df_results.at[i, 'total_tokens'] = str(result.get('total_tokens', ''))
            df_results.at[i, 'prompt_tokens'] = str(result.get('prompt_tokens', ''))
            df_results.at[i, 'completion_tokens'] = str(result.get('completion_tokens', ''))

            # Memory Cleanup
            if torch.cuda.is_available():
                torch.cuda.empty_cache()

            # Checkpoint Saves
            df_results.to_pickle(results_path_pkl)
            df_results.to_excel(results_path_xlsx, index=False)

            time.sleep(1) # Sleep delay for stability

        else:
            print(f"Skipping row {i}, already computed.")

    return df_results

## Populate Long-Term Memory (run once before evaluation)

In [ ]:
# Populate the long-term memory graph
# Run this cell only when initializing/rebuilding the memory dataset

df_memory_results = evaluate_model(
    run_name=f"{MODEL_LABEL}_MEMORY_POPULATION",
    output_name=f"{MODEL_FILE_LABEL}_memory_population",
    configuration="temp",
    model=model,
    target_file=MEMORY_DATASET,
    output_root=OUTPUT_ROOT,
    experiment=EXPERIMENT,
    save_to_memory=True
)

## Evaluation 

In [ ]:
# Run evaluation
# Long-term memory writing is disabled for the test set

df_results = evaluate_model(
    run_name=RUN_NAME,
    output_name=OUTPUT_NAME,
    configuration=CONFIGURATION,
    model=model,
    target_file=TEST_DATASET,
    output_root=OUTPUT_ROOT,
    experiment=EXPERIMENT,
    save_to_memory=False
)